In [ ]:
#/usr/bin/env python3

import pandas as pd 
import numpy as np
import scanpy as sc
import infercnvpy as cnv
import os
import copy
import math
import pickle
import warnings
from plotnine import *
from kneed import KneeLocator
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.pyplot import rc_context
import anndata as ad

# UPDATE: Set to your output directory
outdir = "results/02_prep_infercnv"
if not os.path.exists(outdir): os.makedirs(outdir)

This notebook loads individual `cellranger count` outputs. Each sample's `filtered_feature_bc_matrix.h5`
is loaded independently, annotated with metadata, and concatenated into a single AnnData object.
Cells and genes are then filtered.

In [ ]:
warnings.simplefilter("ignore")

SAMPLES_INFO = Path("./260424_aggregated_samples_info_paper.csv")

In [ ]:
# UPDATE: Set to your genes GTF annotation file path
GENOME_GTF_PATH = Path("path/to/genes.gtf.gz")

# Make an annotation table from the GTF
gene_info = pd.read_csv(GENOME_GTF_PATH, sep="\t", comment='#', header=None)
gene_info = gene_info[gene_info[2] == 'gene'] 
gene_info['gene_ids'] = gene_info[8].str.extract(r'gene_id "(.*?)"', expand=False)
gene_info['gene_ids'] = gene_info['gene_ids'].str.replace(r'\.\d+', '', regex=True)
gene_info['gene_name'] = gene_info[8].str.extract(r'gene_name "(.*?)"', expand=False)
gene_info['chromosome'] = gene_info[0]
gene_info['start'] = gene_info[3].astype(int)
gene_info['end'] = gene_info[4].astype(int)
gene_info = gene_info[['gene_ids', 'gene_name', 'chromosome', 'start', 'end']]

In [ ]:
samples_info = pd.read_csv(SAMPLES_INFO, index_col="Sample_Name")
print(f"Loaded {samples_info.shape[0]} samples")
samples_info.head()

## Load individual cellranger count outputs

Read each sample's `filtered_feature_bc_matrix.h5` from the path stored in the
`Cellranger_counts_h5` column (replacing `molecule_info.h5` with `filtered_feature_bc_matrix.h5`),
annotate with metadata, and concatenate.

In [ ]:
adatas = []
failed = []
varnames = None
for sample_name, row in samples_info.iterrows():
    h5_path = Path(str(row['Cellranger_counts_h5']).replace('molecule_info.h5', 'filtered_feature_bc_matrix.h5'))
    
    if not h5_path.exists():
        print(f"WARNING: File not found for {sample_name}: {h5_path}")
        failed.append(sample_name)
        continue
    
    adata = sc.read_10x_h5(h5_path)
    if varnames is None:
        varnames = adata.var_names
    else:
        assert adata.var_names.equals(varnames), "Not the same genes"
        
    adata.var_names_make_unique()
    
    # Prefix barcodes with sample name to ensure uniqueness across samples
    adata.obs_names = sample_name + "_" + adata.obs_names.astype(str)
    adata.obs['cell_barcode'] = adata.obs_names
    adata.obs['sample_id'] = sample_name
    adata.obs['Timepoint'] = row['Timepoint']
    adata.obs['Treatment'] = row['Treatment']
    adata.obs['Donor'] = row['Donor']
    adata.obs['Run'] = row['Run']
    
    adatas.append(adata)
    print(f"Loaded {sample_name}: {adata.shape[0]} cells, {adata.shape[1]} genes")

print(f"\nSuccessfully loaded {len(adatas)} / {samples_info.shape[0]} samples")
if failed:
    print(f"Failed samples: {failed}")

In [ ]:
full_run = ad.concat(adatas, join='outer', merge='first').copy()
full_run.obs_names_make_unique()
print(f"Concatenated AnnData: {full_run.shape[0]} cells, {full_run.shape[1]} genes")
print(f"Samples: {full_run.obs['sample_id'].nunique()}")
print(f"Treatments: {full_run.obs['Treatment'].unique().tolist()}")
print(f"Timepoints: {full_run.obs['Timepoint'].unique().tolist()}")
print(f"Runs: {full_run.obs['Run'].unique().tolist()}")

In [ ]:
# Add chromosome/start/end annotations to var
full_run.var["gene_ids"] = full_run.var["gene_ids"].str.replace(r'\.\d+', '', regex=True)
original_var_index = full_run.var.index
full_run.var = full_run.var.merge(gene_info, on="gene_ids", how="left")
full_run.var.index = original_var_index
full_run.var_names_make_unique()

print(f"Final annotated AnnData: {full_run.shape}")
full_run.obs.head()

## Plotting functions

In [ ]:
def plot_gene_detection(adata):
    fracs = [0.01, 0.02, 0.05, 0.06, 0.07, 0.1]

    X = adata.X
    if hasattr(X, "toarray"):
        n_cells_per_gene = np.array((X > 0).sum(axis=0)).flatten()
    else:
        n_cells_per_gene = (X > 0).sum(axis=0)

    n_cells = adata.shape[0]
    thresholds = {f: int(f * n_cells) for f in fracs}

    plt.figure(figsize=(8, 4))
    plt.hist(n_cells_per_gene, bins=100, log=True)

    for frac, val in thresholds.items():
        plt.axvline(val, linestyle="--", linewidth=1, label=f"{int(frac*100)}% ({val})")

    plt.xlabel("Number of cells gene is expressed in")
    plt.ylabel("Number of genes (log scale)")
    plt.title("Gene detection distribution")
    plt.legend()
    plt.show()

    print("Thresholds (cells):")
    for frac, val in thresholds.items():
        print(f"{int(frac*100)}% -> {val} cells")

    # --- Elbow detection with kneed ---
    sweep_fracs = np.linspace(0.001, 0.20, 200)
    genes_retained = np.array([(n_cells_per_gene >= f * n_cells).sum() for f in sweep_fracs])

    kn = KneeLocator(
        sweep_fracs, genes_retained,
        curve="convex", direction="decreasing", S=1.0
    )
    elbow_frac = kn.knee
    elbow_cells = int(elbow_frac * n_cells)
    elbow_genes = int(np.interp(elbow_frac, sweep_fracs, genes_retained))

    # Plot the elbow curve
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(sweep_fracs * 100, genes_retained, 'k-', linewidth=2)
    ax.axvline(elbow_frac * 100, color='red', linestyle='--', linewidth=2,
               label=f"Elbow: {elbow_frac*100:.1f}% ({elbow_cells} cells) → {elbow_genes} genes")
    for frac in fracs:
        ax.axvline(frac * 100, color='grey', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.set_xlabel("Min % of cells expressing a gene (threshold)")
    ax.set_ylabel("Number of genes retained")
    ax.set_title("Genes retained vs. detection threshold (elbow method)")
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    print(f"\n*** Elbow point: {elbow_frac*100:.2f}% of cells = {elbow_cells} cells → {elbow_genes} genes retained ***")
    return elbow_frac

In [ ]:
def plot_library_size(adata):
    total_counts = np.array(adata.X.sum(axis=1)).flatten()

    plt.figure(figsize=(8, 4))
    plt.hist(total_counts, bins=100, log=True)
    plt.xlabel("Total counts per cell")
    plt.ylabel("Number of cells (log)")
    plt.title("Library size distribution")
    plt.show()

In [ ]:
def plot_genes_per_cell(adata):
    X = adata.X
    if hasattr(X, "toarray"):
        genes_per_cell = np.array((X > 0).sum(axis=1)).flatten()
    else:
        genes_per_cell = (X > 0).sum(axis=1)

    plt.figure(figsize=(8, 4))
    plt.hist(genes_per_cell, bins=100, log=True)
    plt.xlabel("Genes detected per cell")
    plt.ylabel("Number of cells (log)")
    plt.title("Gene detection per cell")
    plt.show()

In [ ]:
plot_gene_detection(full_run)
plot_library_size(full_run)
plot_genes_per_cell(full_run)

# Filtering

In [ ]:
def preprocess_for_infercnv(anndata_obj):
    """
    Perform filtering of cells and genes by common QC criteria for single cell RNA-seq data

    Parameters:
    anndata_obj (AnnData): An AnnData object representing the results of a `cellranger aggr` run as loaded by `scanpy.read_10x_h5()`

    Returns:
    tuple: (processed AnnData object, DataFrame with filtering statistics per sample)
    """
    # Initialize statistics tracking
    stats_list = []

    def record_stats(adata, filter_step):
        """Record number of cells and genes per sample at current filtering step"""
        for sample in adata.obs['sample_id'].unique():
            sample_cells = adata[adata.obs['sample_id'] == sample]
            stats_list.append({
                'sample_id': sample,
                'filter_step': filter_step,
                'n_cells': sample_cells.shape[0],
                'n_genes': sample_cells.shape[1]
            })

    # Record initial state
    print("initial")
    record_stats(anndata_obj, '0_initial')
    
    plot_gene_detection(anndata_obj)
    plot_library_size(anndata_obj)
    plot_genes_per_cell(anndata_obj)

    # Filter cells - min_genes
    print("min genes")
    sc.pp.filter_cells(anndata_obj, min_genes=200)
    record_stats(anndata_obj, '1_min_genes_200')

    # Filter cells - min_counts
    print("min counts")
    sc.pp.filter_cells(anndata_obj, min_counts=1000)
    record_stats(anndata_obj, '2_min_counts_1000')

    # Filter cells - mitochondrial content
    anndata_obj.var['mt'] = anndata_obj.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(anndata_obj, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    print("mt filter")
    anndata_obj = anndata_obj[anndata_obj.obs.pct_counts_mt < 10, :].copy()
    record_stats(anndata_obj, '3_pct_mt_below_10')

    # Filter genes - min_cells
    print("min cells")
    sc.pp.filter_genes(anndata_obj, min_cells=math.floor(anndata_obj.shape[0] * 0.03))
    record_stats(anndata_obj, '4_gene_min_cells_3pct')
    
    # Filter genes - canonical chromosomes only
    print(f"Before filtering non-canonical chromosomes: {anndata_obj.shape}")
    canonical_chromosomes = [f'chr{i}' for i in range(1, 23)]
    anndata_obj = anndata_obj[:, anndata_obj.var['chromosome'].isin(canonical_chromosomes)].copy()
    print(f"After filtering non-canonical chromosomes: {anndata_obj.shape}")
    record_stats(anndata_obj, '5_canonical_chromosomes_only')

    # Convert stats to DataFrame, pivot to wide format, and save
    stats_df = pd.DataFrame(stats_list)
    stats_wide = stats_df.pivot(index='sample_id', columns='filter_step', values=['n_cells', 'n_genes'])
    stats_wide = stats_wide.swaplevel(axis=1).sort_index(axis=1)
    stats_wide.to_csv(os.path.join(outdir, 'preprocessing_filter_stats.csv'))
    print(f"\nFiltering statistics saved to: {os.path.join(outdir, 'preprocessing_filter_stats.csv')}")

    return anndata_obj, stats_df

full_run, filter_stats = preprocess_for_infercnv(full_run)

In [ ]:
stats_wide = filter_stats.pivot(index='sample_id', columns='filter_step', values=['n_cells', 'n_genes'])
stats_wide = stats_wide.swaplevel(axis=1).sort_index(axis=1)
stats_wide.to_csv(os.path.join(outdir, 'preprocessing_filter_stats.csv'))
print(f"\nFiltering statistics saved to: {os.path.join(outdir, 'preprocessing_filter_stats.csv')}")

In [ ]:
cleaned = full_run.copy()

In [ ]:
# Normalize and log transform the filtered data
full_run.raw = full_run.copy()
sc.pp.normalize_total(full_run, target_sum=1e4)
sc.pp.log1p(full_run)

In [ ]:
with open(os.path.join(outdir,'full_run_preprocessed.pickle'), 'wb') as file:
    pickle.dump(full_run, file)

print(f"Saved preprocessed AnnData to: {os.path.join(outdir, 'full_run_preprocessed.pickle')}")
print(f"Final shape: {full_run.shape}")

In [ ]:
plot_gene_detection(cleaned)
plot_library_size(cleaned)
plot_genes_per_cell(cleaned)